In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path

import polars as pl

pl.Config(set_tbl_cols=1000)

In [3]:
data_files = tuple(
    (Path(os.path.abspath("")).resolve().parents[1] / "data").glob("*.csv")
)
data_files

(PosixPath('/home/runar/github/TDT4225-ex3/data/credits.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/keywords.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/links.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/links_small.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/movies_metadata.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/ratings.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/ratings_small.csv'))

In [4]:
from polars.exceptions import ComputeError

for file in data_files:
    try:
        pl.read_csv(file)
    except ComputeError:
        print(f"{file} had problems parsing")

/home/runar/github/TDT4225-ex3/data/movies_metadata.csv had problems parsing


There's a problem in movies_metadata.csv. Some newlines have not been escaped. Fix implemented in data_cleaning:

In [5]:
from tdt4225_ex3.data_cleaning import get_data

datasets = get_data()

By looking at the raw csv files, we see that some columns are json objects:

- credits.csv: cast, crew
- keywords.csv: keywords
- movies_metadata.csv: belongs_to_collection, genres, production_companies, production_countries, spoken_languages

We want to see if there is a consistent schema:

In [17]:
json_columns = {
    "credits": ("cast", "crew"),
    "keywords": ("keywords",),
    "movies_metadata": (
        "belongs_to_collection",
        "genres",
        "production_companies",
        "production_countries",
        "spoken_languages",
    ),
}

In [20]:
import ast
from collections import Counter

for name, cols in json_columns.items():
    for col in cols:
        layouts = Counter()
        for value in getattr(datasets, name)[col].drop_nulls():
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                parsed = [parsed]
            for d in parsed:
                layouts[str({k: type(v).__name__ for k, v in sorted(d.items())})] += 1

        print(f"{name}.{col}")
        for layout, count in layouts.most_common():
            print(count, layout)
        print()

credits.cast
388618 {'cast_id': 'int', 'character': 'str', 'credit_id': 'str', 'gender': 'int', 'id': 'int', 'name': 'str', 'order': 'int', 'profile_path': 'str'}
173856 {'cast_id': 'int', 'character': 'str', 'credit_id': 'str', 'gender': 'int', 'id': 'int', 'name': 'str', 'order': 'int', 'profile_path': 'NoneType'}

credits.crew
369216 {'credit_id': 'str', 'department': 'str', 'gender': 'int', 'id': 'int', 'job': 'str', 'name': 'str', 'profile_path': 'NoneType'}
95098 {'credit_id': 'str', 'department': 'str', 'gender': 'int', 'id': 'int', 'job': 'str', 'name': 'str', 'profile_path': 'str'}

keywords.keywords
158680 {'id': 'int', 'name': 'str'}

movies_metadata.belongs_to_collection
3237 {'backdrop_path': 'str', 'id': 'int', 'name': 'str', 'poster_path': 'str'}
711 {'backdrop_path': 'NoneType', 'id': 'int', 'name': 'str', 'poster_path': 'str'}
517 {'backdrop_path': 'NoneType', 'id': 'int', 'name': 'str', 'poster_path': 'NoneType'}
26 {'backdrop_path': 'str', 'id': 'int', 'name': 'str',

If we treat None as empty string (which is reasonable), we see that all the columns have a consistent schema!